In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.patheffects as path_effects

from matplotlib.colors import LogNorm
from astropy.coordinates import SkyCoord
from microlensing.new_north_pole import ra_dec_to_new_north_pole
from microlensing.plotting import plot_lightcurve

# Read in object and result dataframes

In [ ]:
objects = pd.read_parquet(
    "../objects/sampled_objects.parquet"
)
lightcurves = pd.read_parquet(
    "../results/kde_labelled_lightcurves/"
    "kde_labelled_lightcurves_batch0.parquet"
)
emt = pd.read_parquet(
    "../results/effective_monitoring_time/"
    "effective_monitoring_time_batch0.parquet"
)
event_rates = pd.read_parquet(
    "../results/event_rate/event_rates.parquet"
)

## Let's first visualize where our population of stars lives

In [ ]:
objects[["alpha", "beta"]] = ra_dec_to_new_north_pole(
    objects[["ra", "dec"]].to_numpy()
)

In [ ]:
fig, ax = plt.subplots(subplot_kw={'projection': 'polar'}, figsize=(8, 8))
sc = ax.scatter(
    np.radians(objects["beta"]),
    objects["alpha"],
    s=0.5,
    )
ax.set_rmax(15)

for label in ax.get_yticklabels():  
    label.set_path_effects([
        path_effects.Stroke(linewidth=2, foreground='white'),  # Outline
        path_effects.Normal()  # Normal rendering
    ])

ax.tick_params(labelsize=12)
plt.show()
fig.savefig("../plots/star_distribution.png", bbox_inches='tight', pad_inches=0.1)

### Each one of those little blue dots is a star! These stars are sampled from a region centered on the Large Magellanic Cloud. To make this plot we transformed their (RA, Dec)s into a new spherical coordinate system: one that is centered on the LMC. Let's look at one of their lightcurves.

In [ ]:
object_id = objects.loc[0, "id"]
lc = lightcurves.loc[lightcurves["objectid"] == object_id]
fig, ax = plt.subplots(figsize=(12, 4))
plot_lightcurve(lc, ax)
ax.set_ylabel("Magnitude", fontsize=16)
ax.set_xlabel("MJD", fontsize=16)
ax.set_title(f"Object {object_id} Lightcurve", fontsize=20)
ax.legend(fontsize=12)
plt.show(fig)
fig.savefig("../plots/example_lightcurve.png", bbox_inches='tight', pad_inches=0.1)

### Neat! Let's look at some of the results we computed using `ETLTask`s

In [ ]:
duration_bins = np.geomspace(1e-4, 1e4, num=50)
x = (duration_bins[1:] + duration_bins[:-1]) / 2
y = emt.mean(axis=0).replace(to_replace=0, value=np.nan)
fig, ax = plt.subplots(figsize=(12, 4))
ax.loglog(x, y)
ax.set_ylabel("EMT (Days)", fontsize=14)
ax.set_xlabel("Event Duration (Days)", fontsize=14)
ax.grid(visible=True)
plt.show(fig)
fig.savefig("../plots/emt.png", bbox_inches='tight', pad_inches=0.1)

### This is the mean effective monitoring time of our population, averaged over all of the stars in our population. It quantifies the survey's sensitivity to string microlensing events of various durations. We used `GoodWindowsTask` and `EffectiveMonitoringTimeTask` to compute this result. Finally, let's check out the event rates computed with `EventRateTask`. 

In [ ]:
plot_df = (
    event_rates
    .xs(6, level="tension_index")
    .merge(objects, left_on="objectid", right_on="id")
)
duration_columns = [f"duration_{i}" for i in range(49)]
plot_df[duration_columns] = (
    plot_df[duration_columns] * np.diff(duration_bins)
)

fig, ax = plt.subplots(subplot_kw={'projection': 'polar'}, figsize=(8, 8))
sc = ax.scatter(
    np.radians(plot_df["beta"]),
    plot_df["alpha"],
    c=plot_df[duration_columns].sum(axis=1),
    cmap="coolwarm",
    s=2,
    norm=LogNorm()
    )
ax.set_rmax(4)
# ax.set_rmax(5)

for label in ax.get_yticklabels():  
    label.set_path_effects([
        path_effects.Stroke(linewidth=2, foreground='white'),
        path_effects.Normal()
    ])

cbar = fig.colorbar(sc, ax=ax, pad=0.1, shrink=0.75)
cbar.set_label(r"String Microlensing Event Rate (Days$^{-1}$)", fontsize=14)
cbar.ax.tick_params(labelsize=12)
ax.tick_params(labelsize=12)
plt.show(fig)
fig.savefig("../plots/event_rate.png", bbox_inches='tight', pad_inches=0.1)

### This looks similar to the first plot we made. The difference is that here the color of each point corresponds to the string microlensing event rate for that star. We've also zoomed in to just 4 degrees around the LMC, rather than 15 degrees. This makes it easier to see the interesting region. Stars farther from the LMC are just blue (not literally blue, but blue in the sense that they have a small event rate and are bad targets for a microlensing search).